# Sesión 8 · Un LLM en Bedrock para clasificar reseñas
**Cómputo en la Nube para ML/AI · Miércoles 11 de noviembre**

Tenemos reseñas de clientes y queremos clasificar su **sentimiento** y su **tema**. Ya sabes hacerlo con un modelo clásico. Hoy preguntamos: ¿cuándo conviene un modelo de lenguaje como servicio (Amazon Bedrock) y qué hay que cuidar al usarlo en la nube?

| Parte | Pregunta | ¿Necesita AWS? |
|---|---|---|
| 1 | ¿Qué tan bueno es el modelo clásico… con texto que no vio? | No |
| 2 | Las reseñas traen **datos personales**: ¿cómo los quitamos antes de mandarlas a un servicio? | Regex: no. Comprehend: sí |
| 3 | ¿Quién puede llamar a qué? Permisos de IAM con mínimo privilegio | No |
| 4 | Bedrock: clasificar, medir exactitud, tokens, latencia y costo | Sí |
| 5 | El prompt como hiperparámetro: zero-shot, few-shot y consistencia | Sí |
| 6 | ¿Qué pasa si NO redactamos? | Sí |

> **Las Partes 1 y 3 corren en cualquier computadora.** Si el Sandbox bloquea un servicio, la celda lo avisa y el resto sigue.
> Todos los datos personales son **inventados** (correos `@example.com`, teléfonos ficticios).

## 0 · Configuración

In [ ]:
import importlib.util, subprocess, sys
for paquete, modulo in [("scikit-learn", "sklearn")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=False)

import os, re, json, math, time, fnmatch, random, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(7); np.random.seed(7)
print("Listo ✔")

### Los datos: reseñas con contacto inventado

Cargamos `reviews.csv` (el dataset del curso). Si no está junto al notebook, se generan reseñas de ejemplo.
A unas 40 reseñas les agregamos un **nombre, un correo y un teléfono ficticios** al final, como pasa en la vida real cuando alguien escribe "cualquier duda, escríbanme a…".

In [ ]:
def sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

RUTA = Path("reviews.csv")
if RUTA.exists():
    reviews = pd.read_csv(RUTA)
else:
    print("⚠️ No encontré reviews.csv junto al notebook; uso reseñas de ejemplo generadas aquí.")
    base = [("Excelente calidad, llegó antes de lo esperado.", [5, 4]), ("Muy buen producto por el precio, volvería a comprar.", [5, 4]),
            ("Producto aceptable, cumple su función básica.", [3]), ("Llegó dañado, mala experiencia con el envío.", [1, 2]),
            ("Dejó de funcionar a la semana de uso.", [1, 2]), ("Superó mis expectativas, lo recomiendo totalmente.", [5])]
    productos = ["Cargador USB-C", "Teclado mecánico", "Mouse ergonómico", "Cable HDMI"]
    g = random.Random(7)
    filas = []
    for i in range(150):
        frase, rs = g.choice(base)
        filas.append((f"R{i+1:04d}", g.choice(productos), g.choice(rs), frase, True))
    reviews = pd.DataFrame(filas, columns=["review_id", "product", "rating", "review_text", "verified_purchase"])

NOMBRES = ["Mariana Solís", "Jorge Ibáñez", "Camila Ortega", "Luis Fernández", "Valeria Cruz",
           "Andrés Montoya", "Paola Rivas", "Héctor Salinas"]
g = random.Random(11)
reviews = reviews.copy()
reviews["texto"] = reviews["review_text"]
for col in ["_nombre", "_correo", "_tel"]:
    reviews[col] = ""
for i in g.sample(range(len(reviews)), 40):
    nombre = g.choice(NOMBRES)
    correo = sin_acentos(nombre).lower().replace(" ", ".") + "@example.com"
    tel = f"55 5550 {g.randint(0, 9999):04d}"
    plantilla = g.choice(["{t} Soy {n}, escríbanme a {c} o llámenme al {tel}.",
                          "{t} Contacto: {c}, tel. {tel}.",
                          "{t} Atentamente, {n} ({tel})."])
    reviews.loc[i, "texto"] = plantilla.format(t=reviews.loc[i, "review_text"], n=nombre, c=correo, tel=tel)
    reviews.loc[i, ["_nombre", "_correo", "_tel"]] = [nombre if "{n}" in plantilla else "", correo if "{c}" in plantilla else "", tel]

reviews["tiene_pii"] = reviews["_tel"] != ""
reviews["verdad"] = reviews["rating"].map(lambda r: "positivo" if r >= 4 else ("neutral" if r == 3 else "negativo"))
print(reviews.shape, "· reseñas con datos personales:", int(reviews["tiene_pii"].sum()))
for t in reviews.loc[reviews["tiene_pii"], "texto"].head(3):
    print("-", t)

---
## Parte 1 · El modelo clásico y el texto que no vio

Entrenamos TF-IDF + regresión logística con las reseñas del curso (etiqueta: ≥4 positivo, 3 neutral, ≤2 negativo) y lo probamos en dos lugares: en reseñas **del mismo dataset** y en **reseñas escritas con otras palabras**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(reviews["review_text"], reviews["verdad"], test_size=0.3, random_state=0, stratify=reviews["verdad"])
vec = TfidfVectorizer(); clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(X_tr), y_tr)
exactitud_mismo = float((clf.predict(vec.transform(X_te)) == y_te).mean())
print(f"Exactitud con reseñas del mismo dataset: {exactitud_mismo:.0%}")
print("Frases distintas en TODO el dataset:", reviews["review_text"].nunique(), "de", len(reviews), "reseñas")

Casi 100%, pero fíjate en la segunda línea: el dataset es **muy repetitivo**, así que el modelo solo memorizó unas cuantas frases. Probemos con reseñas que **nunca vio**, escritas por nosotros (incluyen sarcasmo y opiniones mixtas):

In [ ]:
DIFICILES = [
    ("Me llegó rapidísimo y funciona perfecto, súper contento.", "positivo"),
    ("La caja venía aplastada y el producto no prende.", "negativo"),
    ("Está bien para lo que cuesta, nada del otro mundo.", "neutral"),
    ("Tardó tres semanas en llegar, pero el producto es excelente.", "positivo"),
    ("Qué maravilla, se rompió el primer día. Gracias por nada.", "negativo"),
    ("Cumple, ni más ni menos.", "neutral"),
    ("No se lo recomiendo a nadie, pésima calidad.", "negativo"),
    ("Lo compré para regalo y le encantó a mi mamá.", "positivo"),
    ("El color es distinto al de la foto, pero funciona.", "neutral"),
    ("Increíble relación calidad-precio, ya compré dos más.", "positivo"),
    ("Se calienta muchísimo y huele a quemado, lo devolví.", "negativo"),
    ("Regular. Esperaba algo mejor, aunque no está mal.", "neutral"),
    ("Me atendieron súper mal cuando pregunté por mi pedido.", "negativo"),
    ("Perfecto, justo lo que buscaba.", "positivo"),
    ("Es normal, hace lo que dice.", "neutral"),
    ("Un desastre total, lo peor que he comprado.", "negativo"),
]
dificiles = pd.DataFrame(DIFICILES, columns=["texto", "verdad"])
dificiles["pred_clasico"] = clf.predict(vec.transform(dificiles["texto"]))
exactitud_dificil_lr = float((dificiles["pred_clasico"] == dificiles["verdad"]).mean())
print(f"Exactitud del modelo clásico en texto nuevo: {exactitud_dificil_lr:.0%}  (azar con 3 clases ≈ 33%)")
print(pd.crosstab(dificiles["verdad"], dificiles["pred_clasico"]))

**Qué pasó:** el modelo clásico no falla por ser "malo", falla porque el texto nuevo **no se parece a su entrenamiento** (cambio de distribución). Un modelo de lenguaje preentrenado ya vio muchísimo texto, así que podría generalizar. Pero para usarlo hay que mandar las reseñas a un servicio externo… y las reseñas traen datos personales.

### ✏️ Ejercicio 1
Agrega **10 reseñas tuyas** (con al menos 2 de sarcasmo) a `DIFICILES` y vuelve a medir. ¿Cambia la exactitud del modelo clásico? Guarda tus frases: las usas de nuevo en la Parte 5.

---
## Parte 2 · Datos personales: detectar y redactar antes de enviar

Redactar = reemplazar cada dato personal por su tipo (`[EMAIL]`, `[PHONE]`, `[NAME]`). Primero preparamos la conexión a AWS (si el Sandbox no la deja, todo lo demás sigue).

In [ ]:
import boto3
from botocore.exceptions import ClientError, BotoCoreError

sesion = boto3.Session()
REGION = sesion.region_name or "us-east-1"

def intentar(etiqueta, f):
    try:
        r = f(); print(f"✔ {etiqueta}"); return r
    except (ClientError, BotoCoreError) as e:
        print(f"⚠️ {etiqueta}: {type(e).__name__}: {str(e)[:160]}"); return None

sts = sesion.client("sts", region_name=REGION)
identidad = intentar("Identidad (STS)", lambda: sts.get_caller_identity())
HAY_AWS = identidad is not None
if HAY_AWS:
    print("Rol:", identidad["Arn"].split("/")[1] if "/" in identidad["Arn"] else identidad["Arn"], "· Región:", REGION)

comprehend = sesion.client("comprehend", region_name=REGION)
bedrock = sesion.client("bedrock-runtime", region_name=REGION)

### 2.1 · Expresiones regulares contra Comprehend
Las **regex** encuentran formatos fijos (correos, teléfonos). **Comprehend** (`detect_pii_entities`) usa un modelo de ML y también detecta **nombres**.

In [ ]:
RE_CORREO = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
RE_TEL = re.compile(r"\b\d{2}[ -]?\d{4}[ -]?\d{4}\b")

def detectar_pii_regex(texto):
    return ([(m.start(), m.end(), "EMAIL") for m in RE_CORREO.finditer(texto)] +
            [(m.start(), m.end(), "PHONE") for m in RE_TEL.finditer(texto)])

def detectar_pii_comprehend(texto):
    r = comprehend.detect_pii_entities(Text=texto, LanguageCode="es")
    return [(e["BeginOffset"], e["EndOffset"], e["Type"]) for e in r["Entities"] if e["Score"] >= 0.8]

def unir(spans):
    # Junta spans que se traslapan y deja el primero
    res = []
    for a, b, t in sorted(spans):
        if res and a < res[-1][1]:
            res[-1] = (res[-1][0], max(b, res[-1][1]), res[-1][2])
        else:
            res.append((a, b, t))
    return res

def redactar(texto, spans):
    for a, b, t in sorted(unir(spans), reverse=True):
        texto = texto[:a] + f"[{t}]" + texto[b:]
    return texto

def fugas(serie, df):
    # Cuántas reseñas conservan algún dato personal verdadero
    n = 0
    for texto, (_, fila) in zip(serie, df.iterrows()):
        if any(v and v in texto for v in (fila["_nombre"], fila["_correo"], fila["_tel"])):
            n += 1
    return n

reviews["texto_regex"] = [redactar(t, detectar_pii_regex(t)) for t in reviews["texto"]]
con_pii = int(reviews["tiene_pii"].sum())
print(f"Con regex: reseñas que aún filtran algún dato personal = {fugas(reviews['texto_regex'], reviews)} de {con_pii}")
print("Ejemplo:", reviews.loc[reviews['tiene_pii'], 'texto_regex'].iloc[0])

In [ ]:
reviews["texto_redactado"] = reviews["texto_regex"]          # por defecto, lo que alcanzó la regex
if HAY_AWS:
    try:
        nuevos = []
        for t in reviews["texto"]:
            nuevos.append(redactar(t, detectar_pii_regex(t) + detectar_pii_comprehend(t)))
        reviews["texto_redactado"] = nuevos
        print(f"✔ Con regex + Comprehend: fugas = {fugas(reviews['texto_redactado'], reviews)} de {con_pii}")
        print("Ejemplo:", reviews.loc[reviews['tiene_pii'], 'texto_redactado'].iloc[0])
    except (ClientError, BotoCoreError) as e:
        print("⚠️ Comprehend no disponible en este Sandbox:", type(e).__name__)
        print("   Seguimos con la redacción por regex (los NOMBRES se quedan; fíjate en lo que eso implica más adelante).")
else:
    print("Sin AWS: seguimos con la redacción por regex (los nombres se quedan).")

Fíjate en la diferencia: la regex deja pasar los **nombres**. Con ML (Comprehend) se atrapan. Esto es una instancia del tema de siempre: reglas fijas contra un modelo, y **medir** las fugas es lo que decide.

### ✏️ Ejercicio 2
Agrega a `detectar_pii_regex` un patrón para **otro tipo de dato** (por ejemplo, una CURP o un número de tarjeta de 16 dígitos), prueba con 3 ejemplos inventados y explica qué falso positivo podría tener.

---
## Parte 3 · ¿Quién puede llamar a qué? IAM con mínimo privilegio

Un servicio como Bedrock no se abre a todo el mundo. En AWS lo controlan las **políticas de IAM**. Las reglas del juego:

1. **Todo está denegado por defecto** (negación implícita).
2. Un `Allow` abre un permiso concreto: **acción** + **recurso**.
3. Un `Deny` explícito **siempre gana**.

Construimos un evaluador de juguete con esas reglas para probar políticas sin tocar AWS.

In [ ]:
def _coincide(patrones, valor):
    if isinstance(patrones, str):
        patrones = [patrones]
    return any(fnmatch.fnmatchcase(valor.lower(), p.lower()) for p in patrones)

def evaluar(politicas, accion, recurso):
    permitido = False
    for pol in politicas:
        for st in pol["Statement"]:
            if _coincide(st["Action"], accion) and _coincide(st["Resource"], recurso):
                if st["Effect"] == "Deny":
                    return "DENEGADO (Deny explícito)"
                permitido = True
    return "PERMITIDO" if permitido else "DENEGADO (negación implícita)"

def auditar(pol):
    avisos = []
    for st in pol["Statement"]:
        acc = [st["Action"]] if isinstance(st["Action"], str) else st["Action"]
        res = [st["Resource"]] if isinstance(st["Resource"], str) else st["Resource"]
        if st["Effect"] == "Allow":
            if any(a == "*" or a.endswith(":*") for a in acc): avisos.append(f"Acción demasiado amplia: {acc}")
            if "*" in res: avisos.append("Recurso '*': aplica a TODO")
    return avisos or ["Sin avisos"]


ARN_BUCKET = "arn:aws:s3:::reviews-demo-1234"
ARN_MODELO = "arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-micro-v1:0"

politica_ds = {"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"], "Resource": [f"{ARN_BUCKET}/reviews-redactadas/*"]},
    {"Effect": "Allow", "Action": ["bedrock:InvokeModel"], "Resource": [ARN_MODELO]},
    {"Effect": "Allow", "Action": ["comprehend:DetectPiiEntities"], "Resource": ["*"]},
    {"Effect": "Deny",  "Action": ["s3:DeleteObject"], "Resource": [f"{ARN_BUCKET}/*"]},
]}

pruebas = [
    ("s3:GetObject",           f"{ARN_BUCKET}/reviews-redactadas/a.csv"),
    ("s3:GetObject",           f"{ARN_BUCKET}/reviews-crudas/a.csv"),
    ("s3:DeleteObject",        f"{ARN_BUCKET}/reviews-redactadas/a.csv"),
    ("bedrock:InvokeModel",    ARN_MODELO),
    ("bedrock:InvokeModel",    "arn:aws:bedrock:us-east-1::foundation-model/otro-modelo"),
    ("bedrock:CreateModelCustomizationJob", "*"),
]
print(pd.DataFrame([(a, r.split(":", 5)[-1], evaluar([politica_ds], a, r)) for a, r in pruebas],
                   columns=["acción", "recurso", "resultado"]).to_string(index=False))

politica_peligrosa = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": "*", "Resource": "*"}]}
print("\nAuditoría de la política peligrosa:", auditar(politica_peligrosa))
print("Auditoría de la política del equipo :", auditar(politica_ds))

La política del equipo permite leer **solo las reseñas ya redactadas**, llamar **a un solo modelo** y detectar datos personales; no puede borrar objetos ni entrenar modelos propios (que costarían dinero). El aviso de `Recurso '*'` es esperado: `DetectPiiEntities` no admite permisos por recurso, así que ahí no se puede acotar más.

> **Las credenciales también son un secreto.** Una *access key* subida por error a GitHub es una puerta abierta. Por eso en AWS se prefieren **roles** con credenciales temporales (en el Learner Lab, el `LabRole`) y no llaves permanentes.

### ✏️ Ejercicio 3
Escribe una política de **mínimo privilegio** para un rol de auditoría que solo pueda **listar** el bucket y **leer** `reviews-redactadas/`, pero **no** pueda llamar a Bedrock. Pruébala con `evaluar` en al menos 5 casos y confirma que `auditar` no da avisos.

---
## Parte 4 · Bedrock: clasificar con un LLM y medirlo

Bedrock da acceso a modelos de lenguaje por API. La API `converse` usa el mismo formato para todos los modelos. Según la documentación de AWS, las entradas y salidas **no se comparten con los proveedores de los modelos** ni se usan para entrenarlos; aun así, mandar datos personales a un servicio externo es una decisión de **gobernanza**: **redacta antes de enviar**.

In [ ]:
CANDIDATOS = ["amazon.nova-micro-v1:0", "us.amazon.nova-micro-v1:0", "amazon.nova-lite-v1:0",
              "us.amazon.nova-lite-v1:0", "anthropic.claude-3-haiku-20240307-v1:0", "amazon.titan-text-lite-v1"]

def invocar(prompt, sistema=None, modelo=None, temperatura=0.0, max_tokens=120):
    kwargs = dict(modelId=modelo or MODELO,
                  messages=[{"role": "user", "content": [{"text": prompt}]}],
                  inferenceConfig={"maxTokens": max_tokens, "temperature": temperatura})
    if sistema:
        kwargs["system"] = [{"text": sistema}]
    t0 = time.perf_counter()
    r = bedrock.converse(**kwargs)
    return r["output"]["message"]["content"][0]["text"], r["usage"], time.perf_counter() - t0

MODELO = None
if HAY_AWS:
    for m in CANDIDATOS:
        try:
            invocar("Responde solo: ok", modelo=m, max_tokens=5)
            MODELO = m; break
        except (ClientError, BotoCoreError) as e:
            print(f"  {m}: {type(e).__name__}")
print("Modelo elegido:", MODELO or "ninguno (Bedrock no disponible; usa el Playground de la consola y toma capturas)")

In [ ]:
SISTEMA = ("Eres un clasificador de reseñas de una tienda en línea. Responde SOLO con un JSON válido con las llaves "
           '"sentimiento" (positivo, neutral o negativo) y "tema" (una o dos palabras: calidad, envío, precio, '
           "funcionamiento, empaque u otro). Sin texto adicional.")

def clasificar(texto, temperatura=0.0, sistema=None):
    t, uso, dt = invocar(f"Reseña: {texto}", sistema=sistema or SISTEMA, temperatura=temperatura)
    m = re.search(r"\{.*\}", t, re.S)
    try:
        d = json.loads(m.group(0)); d["sentimiento"] = str(d.get("sentimiento", "")).lower()
    except (AttributeError, json.JSONDecodeError):
        d = {"sentimiento": "inválido", "tema": ""}
    return d, uso, dt

PRECIO_ENTRADA_1K = None   # USD por 1,000 tokens de entrada: cópialo de la página de precios de Bedrock para tu modelo
PRECIO_SALIDA_1K = None    # USD por 1,000 tokens de salida

def evaluar_llm(df, col_texto, sistema=None):
    filas = []
    for _, f in df.iterrows():
        d, uso, dt = clasificar(f[col_texto], sistema=sistema)
        filas.append({"verdad": f["verdad"], "pred": d["sentimiento"], "tema": d["tema"],
                      "tok_in": uso["inputTokens"], "tok_out": uso["outputTokens"], "seg": dt})
    r = pd.DataFrame(filas)
    return r, float((r["verdad"] == r["pred"]).mean())

muestra = pd.concat([grupo.sample(min(len(grupo), 10), random_state=1) for _, grupo in reviews.groupby("verdad")])
resultados = None
if MODELO:
    resultados, exactitud_llm = evaluar_llm(muestra, "texto_redactado")      # ¡solo texto redactado!
    print(f"Exactitud del LLM en {len(resultados)} reseñas del dataset: {exactitud_llm:.0%}")
    print(pd.crosstab(resultados["verdad"], resultados["pred"]))
    print(f"\nTokens por reseña: {resultados['tok_in'].mean():.0f} de entrada, {resultados['tok_out'].mean():.0f} de salida · "
          f"latencia media {resultados['seg'].mean():.2f} s")
    if PRECIO_ENTRADA_1K is not None and PRECIO_SALIDA_1K is not None:
        c = (resultados["tok_in"].mean() * PRECIO_ENTRADA_1K + resultados["tok_out"].mean() * PRECIO_SALIDA_1K) / 1000
        print(f"Costo por reseña: ${c:.6f} · por 1,000,000 de reseñas: ${c * 1e6:,.0f}")
    else:
        print("Llena PRECIO_ENTRADA_1K y PRECIO_SALIDA_1K para estimar el costo de 1,000,000 de reseñas.")
else:
    print("Bedrock no disponible: usa el Playground de la consola y toma capturas.")

---
## Parte 5 · Texto nuevo, el prompt como hiperparámetro y consistencia

Ahora probamos lo que el modelo clásico no pudo: las reseñas **difíciles** de la Parte 1. Comparamos tres variantes del prompt:
- **Zero-shot:** solo instrucciones (el `SISTEMA` de arriba).
- **Few-shot:** instrucciones más 3 ejemplos resueltos.

In [ ]:
SISTEMA_FEWSHOT = SISTEMA + """

Ejemplos:
Reseña: Llegó tarde y la caja estaba rota. -> {"sentimiento": "negativo", "tema": "envío"}
Reseña: Hace lo que dice, sin más. -> {"sentimiento": "neutral", "tema": "funcionamiento"}
Reseña: Qué buena compra, mejor de lo que esperaba. -> {"sentimiento": "positivo", "tema": "calidad"}
"""

filas = [{"variante": "TF-IDF + regresión logística", "exactitud": f"{exactitud_dificil_lr:.0%}", "tokens_entrada": "-", "seg_por_reseña": "~0.0001"}]
if MODELO:
    for nombre, sis in [("LLM zero-shot", SISTEMA), ("LLM few-shot", SISTEMA_FEWSHOT)]:
        r, ex = evaluar_llm(dificiles, "texto", sistema=sis)
        filas.append({"variante": nombre, "exactitud": f"{ex:.0%}", "tokens_entrada": f"{r['tok_in'].mean():.0f}", "seg_por_reseña": f"{r['seg'].mean():.2f}"})
        dificiles["pred_" + nombre.split()[-1]] = r["pred"].values
print(pd.DataFrame(filas).to_string(index=False))
if MODELO:
    print("\nReseñas donde el LLM zero-shot se equivocó:")
    print(dificiles.loc[dificiles["pred_zero-shot"] != dificiles["verdad"], ["texto", "verdad", "pred_zero-shot"]].to_string(index=False))
else:
    print("\nRequiere Bedrock.")

Observa dos cosas: los **ejemplos** del prompt cuestan tokens en **cada** llamada (el few-shot es más caro), y el LLM tampoco es perfecto con el sarcasmo. Medir es lo que decide.

### La temperatura y la consistencia
Con `temperatura = 0` el modelo debería dar casi siempre la misma respuesta. En producción necesitas **consistencia**.

In [ ]:
if MODELO:
    pocas = muestra.head(8)
    for temp in [0.0, 1.0]:
        cambios = 0
        for _, f in pocas.iterrows():
            tres = [clasificar(f["texto_redactado"], temperatura=temp)[0]["sentimiento"] for _ in range(3)]
            cambios += len(set(tres)) > 1
        print(f"Temperatura {temp}: reseñas donde el modelo NO fue consistente en 3 intentos: {cambios} de {len(pocas)}")
else:
    print("Requiere Bedrock.")

### ✏️ Ejercicio 4
Cambia el prompt (otras categorías, otros ejemplos, otro formato de salida) y vuelve a medir sobre `dificiles` **y** tus 10 reseñas del Ejercicio 1. Reporta exactitud, tokens de entrada y costo estimado. ¿Cuándo compensa pagar por tokens extras?

### Resumen: ¿LLM o modelo clásico?

In [ ]:
resumen = pd.DataFrame({
    "TF-IDF + regresión logística": ["Necesita cientos de reseñas etiquetadas", "≈ $0 (CPU)", "Milisegundos o menos", "Falla con texto distinto al de entrenamiento", "Alta: mismos datos, misma salida"],
    "LLM en Bedrock":               ["Ninguna (basta un prompt)", "Por token (ver precios)", "Cientos de ms o más", "Generaliza mejor, pero no es perfecto", "Depende de temperatura y prompt"]},
    index=["Datos etiquetados", "Costo por predicción", "Latencia", "Texto nuevo", "Consistencia"]).T
print(resumen.to_string())

---
## Parte 6 · ¿Qué pasa si NO redactamos?
Mandamos reseñas **crudas** y pedimos que extraiga datos de contacto. El modelo puede hacerlo (los datos están ahí). Con el texto redactado ya no hay nada que extraer. (Los datos son inventados.)

In [ ]:
if MODELO:
    ej = reviews[reviews["tiene_pii"]].head(2)
    for _, f in ej.iterrows():
        for etiqueta, texto in [("CRUDO", f["texto"]), ("REDACTADO", f["texto_redactado"])]:
            r, _, _ = invocar(f"Del siguiente texto, extrae nombre, correo y teléfono de quien escribe, si aparecen:\n{texto}", max_tokens=80)
            print(f"[{etiqueta}] {r.strip()[:140]}")
        print()
else:
    print("Requiere Bedrock.")

---
## 💬 Preguntas para discutir

1. El modelo clásico sacó casi 100% en el dataset del curso y mucho menos con tus reseñas nuevas. ¿Por qué? ¿Qué te dice eso sobre el dataset?
2. ¿Qué detectó Comprehend (o qué detectaría) que las regex no? ¿Y al revés? (Si no tuviste Comprehend, razona qué datos se te quedaron.)
3. En tu política de IAM, ¿qué permisos le quitaste a un rol de auditoría y por qué?
4. ¿Qué diferencia hubo entre zero-shot y few-shot en exactitud y en costo?
5. ¿Qué cambió al subir la temperatura? ¿Cuándo importa la consistencia?
6. ¿Qué pasó al mandar texto crudo frente a texto redactado?
7. Para clasificar reseñas, ¿cuándo usarías un LLM de Bedrock y cuándo un modelo clásico? Considera costo, etiquetas, privacidad y consistencia.